In [0]:
"""
Robô de Previsão de Estiagem em Polos Hidrelétricos (Versão Databricks)
Objetivo: Ler as cidades geradas pelo robô da ANEEL, buscar as coordenadas e
          coletar a previsão de chuva para os próximos 16 dias.
"""

import os
import time
import requests
import pandas as pd

# Caminhos absolutos baseados na estrutura do seu Workspace no Databricks
PASTA_SAIDA = "/Workspace/Users/Groups/mba/MBA_Eng_Dados_TurmaG_Energia_Solar/src/dados_tratados"
ARQUIVO_CIDADES_ENTRADA = f"{PASTA_SAIDA}/cidades_com_usina_hidreletrica.csv"
ARQUIVO_PREVISAO_SAIDA = f"{PASTA_SAIDA}/alerta_estiagem_polos_hidreletricos.csv"

# Quantidade de cidades principais que vamos analisar (para não sobrecarregar a API gratuita)
TOP_N_CIDADES = 30


def buscar_coordenadas(municipio: str, uf: str):
    """
    Usa a API de Geocoding do Open-Meteo para encontrar Latitude e Longitude.
    """
    url_geo = f"https://geocoding-api.open-meteo.com/v1/search?name={municipio}&count=1&language=pt&format=json"
    
    try:
        resposta = requests.get(url_geo)
        dados = resposta.json()
        
        if "results" in dados and len(dados["results"]) > 0:
            resultado = dados["results"][0]
            return resultado["latitude"], resultado["longitude"]
    except Exception as e:
        print(f"Erro ao buscar coordenadas de {municipio}-{uf}: {e}")
        
    return None, None


def buscar_previsao_chuva(lat: float, lon: float):
    """
    Busca a previsão de precipitação total (chuva) para os próximos 16 dias 
    usando a API de previsão climática do Open-Meteo.
    """
    url_clima = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}&daily=precipitation_sum&timezone=America/Sao_Paulo&forecast_days=16"
    )
    
    try:
        resposta = requests.get(url_clima)
        dados = resposta.json()
        
        if "daily" in dados and "precipitation_sum" in dados["daily"]:
            chuvas_diarias = dados["daily"]["precipitation_sum"]
            # Remove valores nulos caso a API falhe em algum dia específico
            chuvas_diarias = [c for c in chuvas_diarias if c is not None]
            
            # Retorna a soma de chuva para os 16 dias
            soma_chuva_16_dias = sum(chuvas_diarias)
            return soma_chuva_16_dias
    except Exception as e:
        print(f"Erro ao buscar clima para Lat:{lat} Lon:{lon} - {e}")
        
    return None


def classificar_risco_estiagem(chuva_acumulada_mm: float):
    """
    Classificação simples baseada no volume de chuva quinzenal esperado.
    """
    if chuva_acumulada_mm < 15:
        return "ALTO RISCO DE SECA (Alerta Bandeira Vermelha)"
    elif chuva_acumulada_mm < 50:
        return "ATENÇÃO (Volume Abaixo do Ideal)"
    else:
        return "NORMAL (Seguro)"


def main():
    if not os.path.exists(ARQUIVO_CIDADES_ENTRADA):
        print(f"Erro: O arquivo não foi encontrado no caminho:\n{ARQUIVO_CIDADES_ENTRADA}")
        return

    print("Carregando base de cidades com hidrelétricas do Workspace...")
    df_cidades = pd.read_csv(ARQUIVO_CIDADES_ENTRADA)

    # Ordena pelas cidades com maior Potência Outorgada Total (kw) e pega as TOP_N
    df_cidades = df_cidades.sort_values(by="PotenciaOutorgadaTotalKw", ascending=False)
    df_alvo = df_cidades.head(TOP_N_CIDADES).copy()

    resultados = []

    print(f"\nIniciando busca meteorológica para os {TOP_N_CIDADES} maiores polos de geração...")
    
    for indice, linha in df_alvo.iterrows():
        municipio = linha["Municipio"]
        uf = linha["UF"]
        potencia = linha["PotenciaOutorgadaTotalKw"]
        
        print(f"-> Analisando: {municipio} - {uf} (Potência: {potencia:,.0f} kW)")
        
        # 1. Achar a latitude e longitude
        lat, lon = buscar_coordenadas(municipio, uf)
        
        # Pausa para evitar bloqueio da API
        time.sleep(0.5)
        
        if lat is not None and lon is not None:
            # 2. Buscar a chuva acumulada dos próximos 16 dias
            chuva_16_dias = buscar_previsao_chuva(lat, lon)
            time.sleep(0.5)
            
            if chuva_16_dias is not None:
                risco = classificar_risco_estiagem(chuva_16_dias)
                
                resultados.append({
                    "UF": uf,
                    "Municipio": municipio,
                    "Latitude": lat,
                    "Longitude": lon,
                    "PotenciaUsinas_Kw": potencia,
                    "Previsao_Chuva_16Dias_mm": chuva_16_dias,
                    "Risco_Estiagem": risco
                })

    df_final = pd.DataFrame(resultados)
    
    # Salva o arquivo CSV com o Alerta na mesma pasta dados_tratados
    df_final.to_csv(ARQUIVO_PREVISAO_SAIDA, index=False, encoding="utf-8-sig")
    
    print("\n" + "="*60)
    print("CONCLUÍDO! Robô meteorológico finalizou a varredura.")
    print(f"Arquivo gerado salvo em: {ARQUIVO_PREVISAO_SAIDA}")
    print("="*60)
    
    print("\n[Resumo Rápido dos Polos com Alto Risco de Seca]:")
    alertas = df_final[df_final["Risco_Estiagem"].str.contains("ALTO RISCO")]
    if alertas.empty:
        print("Nenhuma das cidades analisadas está em alto risco de seca severa nos próximos 16 dias.")
    else:
        print(alertas[["UF", "Municipio", "Previsao_Chuva_16Dias_mm"]])

if __name__ == "__main__":
    main()

In [0]:
"""
Robô de Previsão de Estiagem e Risco de Bandeira Tarifária (Versão Databricks)
Objetivo: Analisar os 30 maiores polos hidrelétricos cruzando a previsão 
          de curto prazo (16 dias) com o volume histórico esperado para o 
          mês corrente e o próximo mês (Baseline Climatológico).
"""

import os
import time
import datetime
import calendar
import requests
import pandas as pd

# Caminhos absolutos do seu Workspace no Databricks
PASTA_SAIDA = "/Workspace/Users/Groups/mba/MBA_Eng_Dados_TurmaG_Energia_Solar/src/dados_tratados"
ARQUIVO_CIDADES_ENTRADA = f"{PASTA_SAIDA}/cidades_com_usina_hidreletrica.csv"
ARQUIVO_PREVISAO_SAIDA = f"{PASTA_SAIDA}/alerta_estiagem_mensal_polos.csv"

TOP_N_CIDADES = 30

def obter_datas_historicas_meses_alvo():
    """
    Calcula dinamicamente as datas do mês corrente e do próximo mês, 
    mas referentes ao ano passado, para servir de 'Baseline' do que é esperado.
    """
    hoje = datetime.date.today()
    ano_passado = hoje.year - 1
    mes_atual = hoje.month
    
    # Data de início: dia 1º do mês atual, no ano passado
    inicio_hist = datetime.date(ano_passado, mes_atual, 1)
    
    # Calcula qual é o próximo mês
    mes_prox = mes_atual + 1
    ano_prox = ano_passado
    if mes_prox > 12:
        mes_prox = 1
        ano_prox += 1
        
    # Data de fim: último dia do próximo mês, no ano passado
    _, ultimo_dia = calendar.monthrange(ano_prox, mes_prox)
    fim_hist = datetime.date(ano_prox, mes_prox, ultimo_dia)
    
    return inicio_hist.strftime("%Y-%m-%d"), fim_hist.strftime("%Y-%m-%d")


def buscar_coordenadas(municipio: str, uf: str):
    url_geo = f"https://geocoding-api.open-meteo.com/v1/search?name={municipio}&count=1&language=pt&format=json"
    try:
        resposta = requests.get(url_geo)
        dados = resposta.json()
        if "results" in dados and len(dados["results"]) > 0:
            return dados["results"][0]["latitude"], dados["results"][0]["longitude"]
    except Exception as e:
        print(f"Erro ao buscar coordenadas: {e}")
    return None, None


def buscar_previsao_16_dias(lat: float, lon: float):
    url_clima = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}&daily=precipitation_sum&timezone=America/Sao_Paulo&forecast_days=16"
    )
    try:
        resposta = requests.get(url_clima)
        dados = resposta.json()
        if "daily" in dados and "precipitation_sum" in dados["daily"]:
            chuvas = [c for c in dados["daily"]["precipitation_sum"] if c is not None]
            return sum(chuvas)
    except Exception as e:
        print(f"Erro na previsão 16d: {e}")
    return None


def buscar_historico_meses_alvo(lat: float, lon: float, start_date: str, end_date: str):
    """
    Busca o total de chuvas que ocorreu nesses mesmos meses no ano anterior.
    """
    url_hist = (
        f"https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={lat}&longitude={lon}&start_date={start_date}&end_date={end_date}"
        f"&daily=precipitation_sum&timezone=America/Sao_Paulo"
    )
    try:
        resposta = requests.get(url_hist)
        dados = resposta.json()
        if "daily" in dados and "precipitation_sum" in dados["daily"]:
            chuvas = [c for c in dados["daily"]["precipitation_sum"] if c is not None]
            return sum(chuvas)
    except Exception as e:
        print(f"Erro no histórico: {e}")
    return None


def classificar_risco_mensal(chuva_16d_mm: float, chuva_hist_bimestre_mm: float):
    """
    Compara a previsão de curto prazo com a expectativa do bimestre (mês atual + próximo).
    Se a previsão de 16 dias for menor que 10% do esperado para os dois meses, acende alerta.
    """
    if chuva_hist_bimestre_mm == 0:
        return "INCONCLUSIVO (Sem histórico)"
        
    proporcao = chuva_16d_mm / chuva_hist_bimestre_mm
    
    if proporcao < 0.10:
        return "ALTO RISCO DE SECA (Alerta Bandeira Vermelha)"
    elif proporcao < 0.25:
        return "ATENÇÃO (Risco Moderado)"
    else:
        return "NORMAL (Seguro)"


def main():
    if not os.path.exists(ARQUIVO_CIDADES_ENTRADA):
        print(f"Erro: O arquivo não foi encontrado:\n{ARQUIVO_CIDADES_ENTRADA}")
        return

    # Define o período histórico que usaremos como base (Mês atual + Próximo)
    start_hist, end_hist = obter_datas_historicas_meses_alvo()
    print(f"Baseline Climatológico configurado para o período: {start_hist} a {end_hist}")

    df_cidades = pd.read_csv(ARQUIVO_CIDADES_ENTRADA)
    df_cidades = df_cidades.sort_values(by="PotenciaOutorgadaTotalKw", ascending=False)
    df_alvo = df_cidades.head(TOP_N_CIDADES).copy()

    resultados = []
    print(f"\nIniciando busca meteorológica avançada para os {TOP_N_CIDADES} maiores polos...")
    
    for indice, linha in df_alvo.iterrows():
        municipio = linha["Municipio"]
        uf = linha["UF"]
        potencia = linha["PotenciaOutorgadaTotalKw"]
        
        print(f"-> Analisando: {municipio}-{uf} (Potência: {potencia:,.0f} kW)")
        lat, lon = buscar_coordenadas(municipio, uf)
        time.sleep(0.3) # Evitar bloqueio (Rate Limit)
        
        if lat is not None and lon is not None:
            # Coleta dados das duas APIs (Previsão + Histórico Climatológico)
            chuva_16d = buscar_previsao_16_dias(lat, lon)
            time.sleep(0.3)
            
            chuva_historica = buscar_historico_meses_alvo(lat, lon, start_hist, end_hist)
            time.sleep(0.3)
            
            if chuva_16d is not None and chuva_historica is not None:
                risco = classificar_risco_mensal(chuva_16d, chuva_historica)
                
                resultados.append({
                    "UF": uf,
                    "Municipio": municipio,
                    "PotenciaUsinas_Kw": potencia,
                    "Previsao_Chuva_16Dias_mm": round(chuva_16d, 1),
                    "Expectativa_Chuva_Bimestre_mm": round(chuva_historica, 1),
                    "Risco_Bandeira_Tarifaria": risco
                })

    df_final = pd.DataFrame(resultados)
    df_final.to_csv(ARQUIVO_PREVISAO_SAIDA, index=False, encoding="utf-8-sig")
    
    print("\n" + "="*60)
    print("CONCLUÍDO! Pipeline meteorológico executado com sucesso.")
    print(f"Dataset salvo em: {ARQUIVO_PREVISAO_SAIDA}")
    print("="*60)
    
    print("\n[Cidades com ALTO RISCO de Seca - Gatilho para Bandeira Vermelha]:")
    alertas = df_final[df_final["Risco_Bandeira_Tarifaria"].str.contains("ALTO RISCO")]
    if alertas.empty:
        print("Nenhuma cidade em estado crítico imediato.")
    else:
        print(alertas[["UF", "Municipio", "Previsao_Chuva_16Dias_mm", "Expectativa_Chuva_Bimestre_mm"]])

if __name__ == "__main__":
    main()

In [0]:
import os
import time
import datetime
import requests
import pandas as pd

# Caminhos absolutos do Databricks
PASTA_SAIDA = "/Workspace/Users/Groups/mba/MBA_Eng_Dados_TurmaG_Energia_Solar/src/dados_tratados"
ARQUIVO_CIDADES = f"{PASTA_SAIDA}/cidades_com_usina_hidreletrica.csv"

# Foco nos 15 maiores polos do Brasil (maior peso no Sistema Interligado Nacional)
TOP_N_CIDADES = 15 

def buscar_coordenadas(municipio: str, uf: str):
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={municipio}&count=1&language=pt&format=json"
    try:
        resp = requests.get(url).json()
        if "results" in resp and len(resp["results"]) > 0:
            return resp["results"][0]["latitude"], resp["results"][0]["longitude"]
    except: 
        pass
    return None, None

def buscar_estimativa_setembro_2026(lat: float, lon: float):
    hoje = datetime.date.today()
    ontem = hoje - datetime.timedelta(days=1)
    primeiro_dia_mes = datetime.date(hoje.year, hoje.month, 1)

    # 1. Realizado: 01/09/2026 até ontem
    url_realizado = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={primeiro_dia_mes}&end_date={ontem}&daily=precipitation_sum&timezone=America/Sao_Paulo"
    
    # 2. Previsão: Hoje + 15 dias
    url_prev = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=precipitation_sum&timezone=America/Sao_Paulo&forecast_days=16"
    
    # 3. Baseline: Setembro de 2025 completo
    url_hist = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date=2025-09-01&end_date=2025-09-30&daily=precipitation_sum&timezone=America/Sao_Paulo"
    
    chuva_realizada = 0.0
    chuva_prevista = 0.0
    chuva_historica = 0.0
    
    try:
        r_real = requests.get(url_realizado).json()
        if "daily" in r_real and "precipitation_sum" in r_real["daily"]:
            chuva_realizada = sum([c for c in r_real["daily"]["precipitation_sum"] if c is not None])
            
        r_prev = requests.get(url_prev).json()
        if "daily" in r_prev and "precipitation_sum" in r_prev["daily"]:
            chuva_prevista = sum([c for c in r_prev["daily"]["precipitation_sum"] if c is not None])
            
        r_hist = requests.get(url_hist).json()
        if "daily" in r_hist and "precipitation_sum" in r_hist["daily"]:
            chuva_historica = sum([c for c in r_hist["daily"]["precipitation_sum"] if c is not None])
    except:
        pass
        
    # Extrapola os dias coletados (~22 dias) para estimar o mês cheio de 30 dias
    dias_coletados = (ontem - primeiro_dia_mes).days + 1 + 16
    estimativa_mes_fechado = (chuva_realizada + chuva_prevista) * (30 / dias_coletados)
    
    return estimativa_mes_fechado, chuva_historica

def main():
    if not os.path.exists(ARQUIVO_CIDADES):
        print("ERRO: Arquivo base de cidades não encontrado.")
        return

    df = pd.read_csv(ARQUIVO_CIDADES)
    df = df.sort_values(by="PotenciaOutorgadaTotalKw", ascending=False).head(TOP_N_CIDADES)

    potencia_total_analisada = df["PotenciaOutorgadaTotalKw"].sum()
    potencia_em_seca_severa = 0.0

    print("Processando telemetria climática dos polos principais. Aguarde...\n")
    
    for _, linha in df.iterrows():
        lat, lon = buscar_coordenadas(linha["Municipio"], linha["UF"])
        time.sleep(0.3) 
        
        if lat and lon:
            estimativa_atual, historica = buscar_estimativa_setembro_2026(lat, lon)
            time.sleep(0.3)
            
            # Gatilho: Estimativa final do mês é menor que 40% do volume histórico normal?
            if historica > 0 and (estimativa_atual / historica) < 0.40:
                potencia_em_seca_severa += linha["PotenciaOutorgadaTotalKw"]

    percentual_afetado = (potencia_em_seca_severa / potencia_total_analisada) * 100
    
    print("="*40)
    if percentual_afetado > 50.0:
        print("BANDEIRA VERMELHA OU ACIMA: SIM")
    else:
        print("BANDEIRA VERMELHA OU ACIMA: NÃO")
    print("="*40)
    print(f"-> Nível de estresse hídrico do parque gerador estimado para Setembro/26: {percentual_afetado:.1f}%")

if __name__ == "__main__":
    main()